# Pokémon Battle Prediction 2026 — Competitive XGBoost Pipeline
**Sapienza University of Rome — Foundations of Data Science Competition**

Questo notebook implementa la pipeline completa, riproducibile e conforme alle regole della competizione basata su **XGBoost** per predire il vincitore finale (`player_won` $\in \{0, 1\}$) delle battaglie Pokémon Generation 2 OU a partire dai primi 30 turni osservati (`battle_timeline`).

### Punti Chiave della Metodologia
1. **Conformità Rigorosa alle Regole**: Utilizza esclusivamente alberi di decisione con gradient boosting (`XGBClassifier`), i file JSONL ufficiali della competizione e la tabella dei tipi di Generazione 2 (`type_chart_gen2.csv`). Nessuna rete neurale, modello pre-addestrato o database esterno.
2. **Ricostruzione Fisica dello Stato Battaglia Gen 2 (`src/battle_features.py`)**:
   - **De-duplicazione della Cache di `move_details`**: Filtra i turni di switch-in volontario in cui il parser del dataset ripete l'ultima mossa memorizzata in cache dal Pokémon, evitando il doppio conteggio delle mosse offensive e di stato.
   - **Ricostruzione Fisica dei Single-KO del Player 2**: Poiché `p2_pokemon_state.status == "fnt"` compare solo nei doppi KO simultanei e al Turno 30 (quando un Pokémon di P2 va KO da solo nei turni 1–29, il sostituto compare immediatamente nello snapshot di fine turno), ricostruiamo i KO singoli di P2 tramite la fisica di Gen 2 (verifica del sostituto intatto, attacco eseguito da P1, efficacia di tipo, danno residuo da status ed *Explosion*).
   - **Catalogo Specie Interno Within-Fold**: Costruito esclusivamente dai dati di training (`p1_team_details` e `p2_lead_details`, 247 specie con 0 incongruenze) per inferire statistiche base e tipi dei Pokémon rivelati da P2.
3. **Sei Famiglie Modulari di Feature + Target Encoding Leak-Free Within-Fold**:
   - **Family A (`team`)**: Statistiche base del team completo di P1, bulk fisico/speciale, speed tier, diversità di tipi e staple del metagame Gen 2 OU.
   - **Family B (`p2_observed`)**: Statistiche del lead e dei Pokémon rivelati da P2, incertezza sulle riserve non rivelate e categorie di mosse osservate.
   - **Family C (`state`)**: Bilancio materiale al Turno 30, differenziale vivi/fainted, HP pesati per recency, gravità degli status (distinguendo *Rest* dal sonno inflitto dal nemico), boost attivi e phazed-absence score.
   - **Family D (`temporal`)**: Traiettorie multi-finestra (`Turns 1–10`, `11–20`, `21–30`, `Last 5`, `Last 10`), pendenza lineare (momentum), cambi di vantaggio e controllo del ritmo.
   - **Family E (`matchup`)**: Matrice di efficacia dei tipi Gen 2 (`type_chart_gen2.csv`), moltiplicatori STAB attivi, miglior counter disponibile nel roster e Speed/KO threat index.
   - **Family F (`interactions`)**: Rapporti non lineari, vantaggio di sopravvivenza aggiustato per HP effettivi e indice di minaccia *Endgame Sweeper* al Turno 30.
   - **`WithinFoldTargetEncoder`**: Calcola i win-rate empirici smoothed (specie, lead matchup e sopravvissuti pesati per HP al Turno 30) rigorosamente all'interno di ciascun fold tramite 5-fold OOF annidati.
4. **Generazione Automatica dell'Unico File `submission.csv`**: Alla fine del notebook viene generato e validato l'unico file **`submission.csv`** pronto per la submission automatica alla challenge su Kaggle.

## Step 1 & 2: Setup Ambiente e Clonazione Repository GitHub
Su Kaggle il notebook clona automaticamente il repository GitHub e lo aggiunge a `sys.path`. In esecuzione locale rileva direttamente la directory del progetto.

In [37]:
# Standard library imports for filesystem navigation and process execution
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

REPO_URL = os.environ.get("REPO_URL", "https://github.com/danielegabriele17-rgb/predictive-spikes.git")
GIT_REF = os.environ.get("GIT_REF", "main")
CLONE_DIR = Path("/tmp/predictive-spikes")

def setup_repository() -> Path:
    """Locate the repository root, inspect Kaggle inputs, or clone cleanly."""
    # 1. Controlla cartelle locali o dataset Kaggle collegati
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        CLONE_DIR,
        Path("/kaggle/input/predictive-spikes"),
        Path("/kaggle/input/predictive-spikes-code"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "battle_features.py").exists():
            print(f"Repository rilevato in: {candidate.resolve()}")
            return candidate.resolve()

    # 2. Se CLONE_DIR esiste già ma è vuota o corrotta, ripuliscila prima del clone
    if CLONE_DIR.exists():
        print(f"Directory {CLONE_DIR} esistente trovata. Rimozione per clone pulito...")
        shutil.rmtree(CLONE_DIR, ignore_errors=True)

    print(f"Clonazione di {REPO_URL} in {CLONE_DIR}...")
    try:
        res = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(CLONE_DIR)],
            capture_output=True,
            text=True,
            check=True,
        )
        print("Clone completato con successo.")
        return CLONE_DIR.resolve()
    except subprocess.CalledProcessError as e:
        print("\n" + "!" * 75)
        print("ERRORE DURANTE IL GIT CLONE (Exit code 128):")
        print("Standard error:", e.stderr.strip() if e.stderr else "N/A")
        print("Standard output:", e.stdout.strip() if e.stdout else "N/A")
        print("!" * 75)
        if "Could not resolve host" in str(e.stderr) or "Connection refused" in str(e.stderr):
            print("\n>>> SOLUZIONE: In Kaggle, assicurati che 'Internet' sia attivato nel pannello laterale:")
            print("    Notebook Settings (a destra) -> 'Internet: On'.")
        raise e

# Setup repository root e sys.path
REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository Root : {REPO_ROOT}")
print(f"Working Dir     : {Path.cwd().resolve()}")


Repository rilevato in: /Users/Daniele/Documents/GitHub/predictive-spikes
Repository Root : /Users/Daniele/Documents/GitHub/predictive-spikes
Working Dir     : /Users/Daniele/Documents/GitHub/predictive-spikes/notebooks


## Step 3: Import dei Moduli della Pipeline (`src/`)

In [38]:
# Core scientific computing and gradient boosting libraries
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

try:
    from IPython.display import display
except ImportError:
    display = print

# Project pipeline modules for feature extraction, CV validation, and submission generation
from src.battle_features import build_species_catalogue
from src.data_loader import discover_data_dir, load_competition_data
from src.feature_engineering import (
    ALL_FEATURE_FAMILIES,
    WithinFoldTargetEncoder,
    build_feature_dataframe,
    get_default_feature_subsets,
)
from src.model import get_feature_importance_df, optimize_ensemble_weights, run_xgb_cv
from src.submission import generate_submission, validate_submission_df
from src.validation import get_cv_splits

print(f"XGBoost version: {xgb.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

XGBoost version: 3.4.1
NumPy version  : 2.5.4
Pandas version : 3.1.0


## Step 4 & 5: Rilevamento Automatico e Caricamento Dataset
Il data loader (`discover_data_dir` e `load_competition_data`) individua automaticamente `train.jsonl`, `test.jsonl`, `sample_submission.csv` e `type_chart_gen2.csv` sia su Kaggle (`/kaggle/input/...`) sia in locale (`fds-pokemon-battles-prediction-2026/`).

In [39]:
# Automatically discover the dataset directory (local repo or /kaggle/input) and load all files
t_start = time.perf_counter()
DATA_DIR = discover_data_dir()
sample_submission_path = DATA_DIR / "sample_submission.csv"

# Load 10,000 training battles, 5,000 test battles, sample_submission.csv, and Gen 2 type chart
train_battles, test_battles, sample_sub, type_chart = load_competition_data(custom_dir=DATA_DIR)
y = np.array([int(b["player_won"]) for b in train_battles], dtype=int)

print(f"Directory dati attiva : {DATA_DIR}")
print(f"Battaglie di Training : {len(train_battles):,} | Target positive rate: {y.mean():.4f}")
print(f"Battaglie di Test     : {len(test_battles):,} | Sample submission shape: {sample_sub.shape}")
print(f"Gen 2 Type Chart      : {len(type_chart)} coppie attacco-difesa caricate")

Directory dati attiva : /Users/Daniele/Documents/GitHub/predictive-spikes/fds-pokemon-battles-prediction-2026
Battaglie di Training : 10,000 | Target positive rate: 0.5000
Battaglie di Test     : 5,000 | Sample submission shape: (5000, 2)
Gen 2 Type Chart      : 110 coppie attacco-difesa caricate


## Step 6: Estrazione delle 6 Famiglie di Feature e Pokédex Interno
Costruiamo il catalogo interno delle specie esclusivamente da `train.jsonl` ed estraiamo tutte le 6 famiglie modulari di feature (`team`, `p2_observed`, `state`, `temporal`, `matchup`, `interactions`) per le 10,000 battaglie di training e le 5,000 battaglie di test.

In [40]:
# Build internal species catalogue strictly from training data (zero test set leakage)
t_fe = time.perf_counter()
catalogue = build_species_catalogue(train_battles)
print(f"Catalogo interno delle specie costruito dal training set: {len(catalogue)} specie Pokémon registrate")

# Extract all 6 observation-only feature families (A-F) into pandas DataFrames
X_train_full, train_ids, y_extracted = build_feature_dataframe(
    train_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
X_test_full, test_ids, _ = build_feature_dataframe(
    test_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
print(f"Matrici di feature estratte in {time.perf_counter() - t_fe:.2f}s:")
print(f"  X_train_full: {X_train_full.shape}")
print(f"  X_test_full : {X_test_full.shape}")

Catalogo interno delle specie costruito dal training set: 239 specie Pokémon registrate
Matrici di feature estratte in 8.61s:
  X_train_full: (10000, 549)
  X_test_full : (5000, 549)


## Step 7: Configurazione Diretta dei Modelli e degli Iperparametri

Configuriamo i **4 modelli XGBoost dell'Ensemble**, la soglia di decisione e i parametri di CV **direttamente nel notebook in codice Python** (senza dover modificare file YAML esterni).

> **Sperimentazione Interattiva**: puoi modificare direttamente qui sotto parametri come `max_depth` (es. `1`, `3`, `4`, `5`), `learning_rate`, `n_estimators`, `subsample` o la soglia `DECISION_THRESHOLD` (es. `0.50` vs `0.519`) e rieseguire lo Step 8 per vederne subito l'effetto!

In [41]:
# ==============================================================================
# 1. PARAMETRI GENERALI DI CROSS-VALIDATION E SOGLIA DI DECISIONE
# ==============================================================================
RANDOM_SEED = 42
# FINAL_FOLDS = 5
FINAL_FOLDS = 10
# DECISION_THRESHOLD = 0.50
DECISION_THRESHOLD = 0.50

# Sottoinsiemi di feature selezionati (pruned: 511, top200: 200, top220: 220, top140: 140)
feature_subsets = get_default_feature_subsets()

# ==============================================================================
# 2. CONFIGURAZIONE ORIGINALE VALIDATA DEI 4 MODELLI XGBOOST
# ==============================================================================
models_cfg = {
    "model1_optuna_top200_te": {
        "description": "Optuna-tuned XGBoost on Top-200 features + Within-Fold TE",
        "feature_subset": "top200_features",
        "use_within_fold_te": True,
        # "early_stopping_rounds": 50,
        "early_stopping_rounds": None,
        
        "xgb_params": {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "n_estimators": 800,
            "learning_rate": 0.0384,
            "max_depth": 5,
            "min_child_weight": 20,
            "subsample": 0.794,
            "colsample_bytree": 0.680,
            "colsample_bylevel": 0.645,
            "gamma": 0.052,
            "reg_alpha": 0.0096,
            "reg_lambda": 0.457,
            "random_state": RANDOM_SEED,
            "n_jobs": 4,
        },
    },
    "model2_conservative_depth4_pruned_te": {
        "description": "Conservative shallow (depth=4) regularized XGBoost on 511 features + Within-Fold TE",
        "feature_subset": "pruned_features",
        "use_within_fold_te": True,
        # "early_stopping_rounds": 50,
        "early_stopping_rounds": None,
        "xgb_params": {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "n_estimators": 550,
            "learning_rate": 0.040,
            # "max_depth": 4,
            "max_depth": 5,
            "min_child_weight": 15,
            "subsample": 0.750,
            "colsample_bytree": 0.450,
            "colsample_bylevel": 0.750,
            "gamma": 0.400,
            "reg_alpha": 1.500,
            "reg_lambda": 5.000,
            "random_state": RANDOM_SEED,
            "n_jobs": 4,
        },
    },
    "model3_obs_only_top220_no_te": {
        "description": "Pure Observation-Only (Zero-TE) XGBoost on Top-220 features (Hedge vs TE Drift)",
        "feature_subset": "top220_features",
        "use_within_fold_te": False,
        # "early_stopping_rounds": 50,
        "early_stopping_rounds": None,
        "xgb_params": {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "n_estimators": 500,
            "learning_rate": 0.040,
            # "max_depth": 5,
            "max_depth": 5,
            "min_child_weight": 12,
            "subsample": 0.780,
            "colsample_bytree": 0.550,
            "colsample_bylevel": 0.800,
            "gamma": 0.250,
            "reg_alpha": 0.800,
            "reg_lambda": 3.500,
            "random_state": RANDOM_SEED,
            "n_jobs": 4,
        },
    },
    "model4_compact_top140_te": {
        "description": "Compact Top-140 features + Within-Fold TE with depth-5 regularized trees",
        "feature_subset": "top140_features",
        "use_within_fold_te": True,
        # "early_stopping_rounds": 50,
        "early_stopping_rounds": None,
        "xgb_params": {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "n_estimators": 500,
            "learning_rate": 0.032,
         #  "max_depth": 4,
            "max_depth": 5,
            "min_child_weight": 10,
            "subsample": 0.800,
            "colsample_bytree": 0.600,
            "colsample_bylevel": 0.800,
            "gamma": 0.200,
            "reg_alpha": 0.500,
            "reg_lambda": 2.500,
            "random_state": RANDOM_SEED,
            "n_jobs": 4,
        },
    },
}

# ==============================================================================
# 3. CONFIGURAZIONE ENSEMBLE E PESI DI PARTENZA
# ==============================================================================
ens_cfg = {
    "members": [
        "model1_optuna_top200_te",
        "model2_conservative_depth4_pruned_te",
        "model3_obs_only_top220_no_te",
        "model4_compact_top140_te",
    ],
    "weights": [0.15, 0.35, 0.10, 0.40],  # Pesi euristici di partenza (w0)
    "min_weight": 0.10,                     # Floor di diversità (Model 3 >= 10%)
    "shrinkage": 0.20,                      # Shrinkage Bayesiano verso pesi uniformi 1/M
}

print("Configurazione impostata con successo direttamente nel notebook:")
for subset_name, cols in feature_subsets.items():
    print(f"  Feature subset [{subset_name:18s}]: {len(cols)} feature")
print(f"Modelli configurati ({len(models_cfg)}):", list(models_cfg.keys()))
print(f"Decision Threshold: {DECISION_THRESHOLD} | Splits: {FINAL_FOLDS}-Fold Stratified CV")

Configurazione impostata con successo direttamente nel notebook:
  Feature subset [pruned_features   ]: 511 feature
  Feature subset [top200_features   ]: 200 feature
  Feature subset [top220_features   ]: 220 feature
  Feature subset [top140_features   ]: 140 feature
Modelli configurati (4): ['model1_optuna_top200_te', 'model2_conservative_depth4_pruned_te', 'model3_obs_only_top220_no_te', 'model4_compact_top140_te']
Decision Threshold: 0.5 | Splits: 10-Fold Stratified CV


## Step 8 & 9: Addestramento in 5-Fold Stratified CV e Multi-Model Ensemble Blending
Addestriamo i modelli XGBoost sui 5 fold stratificati (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`) calcolando il `WithinFoldTargetEncoder` rigorosamente dentro ciascun fold (zero data leakage) e combiniamo le probabilità OOF e di Test nell'Ensemble finale:
- **`model1_optuna_top200_te`** (`weight = 0.15`): Modello Optuna TPE (`max_depth=4`, `min_child_weight=20`) sulle Top-200 feature + Within-Fold TE.
- **`model2_conservative_depth4_pruned_te`** (`weight = 0.35`): Modello shallow fortemente regolarizzato (`max_depth=4`, `colsample_bytree=0.45`, `reg_lambda=5.0`) su tutte le 512 feature pruned + Within-Fold TE.
- **`model3_obs_only_top220_no_te`** (`weight = 0.10`): Modello *Pure Observation-Only* (Zero Target Encoding, `max_depth=5`) sulle Top-220 feature.
- **`model4_compact_top140_te`** (`weight = 0.40`): Modello compatto ad alto rapporto segnale/rumore (`max_depth=5`, `colsample_bytree=0.60`) sulle Top-140 feature + Within-Fold TE.

In [42]:
# 1. Generate reproducible 5-fold StratifiedKFold splits
cv_splits = get_cv_splits(train_battles, y, n_splits=FINAL_FOLDS, seed=RANDOM_SEED, group_duplicates=False)

# 2. Precompute strictly within-fold out-of-fold (OOF) Bayesian target encodings (zero leakage)
te_folds_train = []
te_folds_val = []
te_folds_test = []
for tr_idx, va_idx in cv_splits:
    enc = WithinFoldTargetEncoder(smoothing=30.0, team_smoothing=20.0)
    tr_b = [train_battles[i] for i in tr_idx]
    va_b = [train_battles[i] for i in va_idx]
    te_folds_train.append(enc.fit_transform_inner_oof(tr_b, y[tr_idx], n_inner_splits=5, seed=RANDOM_SEED))
    te_folds_val.append(enc.transform(va_b))
    te_folds_test.append(enc.transform(test_battles))

fold_map = {(int(tr[0]), int(va[0])): idx for idx, (tr, va) in enumerate(cv_splits)}

def fold_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_train[idx], te_folds_val[idx]

def test_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_test[idx]

# 3. Train and evaluate each of the 4 diverse XGBoost configurations across the 5 folds
oof_probs_map = {}
test_probs_map = {}
cv_results_map = {}

for m_name, m_spec in models_cfg.items():
    cols = feature_subsets[m_spec["feature_subset"]]
    use_te = bool(m_spec.get("use_within_fold_te", True))
    es_rounds = m_spec.get("early_stopping_rounds", 50)
    params = m_spec["xgb_params"]

    cv_res, test_probs = run_xgb_cv(
        X_df=X_train_full[cols],
        y=y,
        splits=cv_splits,
        xgb_params=params,
        experiment_id=m_name,
        early_stopping_rounds=es_rounds,
        fold_feature_fn=fold_te_fn if use_te else None,
        X_test_df=X_test_full[cols],
        test_fold_feature_fn=test_te_fn if use_te else None,
        threshold=DECISION_THRESHOLD,
        seed=RANDOM_SEED,
    )
    cv_results_map[m_name] = cv_res
    oof_probs_map[m_name] = cv_res.oof_probs
    test_probs_map[m_name] = test_probs
    print(
        f"[{m_name:36s}] 5-Fold CV Acc: {cv_res.cv_mean_acc:.4f} +/- {cv_res.cv_std_acc:.4f} "
        f"| ROC-AUC: {cv_res.oof_roc_auc:.4f} | LogLoss: {cv_res.oof_log_loss:.4f} "
        f"| Train-Val Gap: {np.mean(cv_res.fold_train_accs) - cv_res.cv_mean_acc:.4f}"
    )

# 4. Blend probabilities using robust SLSQP weight optimization with diversity floor and shrinkage
# Metodo ingegnerizzato per massimizzare la generalizzazione su dati di test non visti (Private Leaderboard):
# - SLSQP risolve il problema convesso vincolato su log-loss garantendo sum(w) == 1 e w_i >= 0.10 (diversity floor).
# - Il floor w_i >= 10% impedisce all'ottimizzatore di affamare Model 3 (Zero-TE), che rappresenta la nostra
#   principale assicurazione contro il potenziale drift del Target Encoding sul test set nascosto.
# - Lo shrinkage bayesiano (20% verso pesi uniformi 1/M) mitiga il rumore di stima della varianza campionaria.
members = ens_cfg["members"]
heuristic_weights = np.asarray(ens_cfg["weights"], dtype=float)
heuristic_weights = heuristic_weights / np.sum(heuristic_weights)

P_oof = np.column_stack([oof_probs_map[m] for m in members])
P_test = np.column_stack([test_probs_map[m] for m in members])

weights = optimize_ensemble_weights(
    y_true=y,
    oof_predictions=P_oof,
    init_weights=heuristic_weights,
    min_weight=float(ens_cfg.get("min_weight", 0.10)),
    shrinkage=float(ens_cfg.get("shrinkage", 0.20)),
    metric="logloss",
)
print("Pesi Euristici Iniziali     :", [round(x, 4) for x in heuristic_weights])
print("Pesi Ottimizzati Robusti    :", [round(x, 4) for x in weights])

blended_oof_probs = np.dot(P_oof, weights)
blended_test_probs = np.dot(P_test, weights)

ens_fold_accs = [
    float(accuracy_score(y[va_idx], (blended_oof_probs[va_idx] >= DECISION_THRESHOLD).astype(int)))
    for _, va_idx in cv_splits
]
final_oof_acc = float(np.mean(ens_fold_accs))
final_oof_std = float(np.std(ens_fold_accs))
final_oof_auc = float(roc_auc_score(y, blended_oof_probs))
final_oof_loss = float(log_loss(y, blended_oof_probs))

print("=" * 80)
print(f"RISULTATO FINALE ENSEMBLE ({FINAL_FOLDS}-Fold x {len(members)} Configurazioni = {FINAL_FOLDS * len(members)} modelli XGBoost)")
print(f"OOF Validation Accuracy : {final_oof_acc:.4f} +/- {final_oof_std:.4f}")
print(f"OOF ROC-AUC             : {final_oof_auc:.4f}")
print(f"OOF Log-Loss            : {final_oof_loss:.4f}")
print("=" * 80)

[model1_optuna_top200_te             ] 5-Fold CV Acc: 0.6937 +/- 0.0122 | ROC-AUC: 0.7686 | LogLoss: 0.5765 | Train-Val Gap: 0.2348
[model2_conservative_depth4_pruned_te] 5-Fold CV Acc: 0.6925 +/- 0.0139 | ROC-AUC: 0.7675 | LogLoss: 0.5741 | Train-Val Gap: 0.2207
[model3_obs_only_top220_no_te        ] 5-Fold CV Acc: 0.6949 +/- 0.0118 | ROC-AUC: 0.7683 | LogLoss: 0.5728 | Train-Val Gap: 0.1990
[model4_compact_top140_te            ] 5-Fold CV Acc: 0.6988 +/- 0.0139 | ROC-AUC: 0.7737 | LogLoss: 0.5665 | Train-Val Gap: 0.1699
Pesi Euristici Iniziali     : [np.float64(0.15), np.float64(0.35), np.float64(0.1), np.float64(0.4)]
Pesi Ottimizzati Robusti    : [np.float64(0.13), np.float64(0.13), np.float64(0.1936), np.float64(0.5464)]
RISULTATO FINALE ENSEMBLE (10-Fold x 4 Configurazioni = 40 modelli XGBoost)
OOF Validation Accuracy : 0.7005 +/- 0.0122
OOF ROC-AUC             : 0.7747
OOF Log-Loss            : 0.5657


## Step 10: Perché Generiamo Direttamente un Unico `submission.csv` (Scelta del Modello Finale)

Durante la fase sperimentale abbiamo testato e confrontato in **5-Fold Stratified Cross-Validation** l'esportazione separata dei singoli modelli candidati (precedentemente indicati come *Submission 1* basata sul singolo `model4_compact_top140_te` e *Submission 2* basata sul singolo `model2_conservative_depth4_pruned_te`) rispetto al **Multi-Model Diverse XGBoost Ensemble**:

| Candidato Testato | 5-Fold OOF Accuracy | Deviazione Std Fold | OOF ROC-AUC | OOF Log-Loss | Esito |
| :--- | :---: | :---: | :---: | :---: | :--- |
| `model1_optuna_top200_te` (Singolo) | `0.6970` | `± 0.0022` | `0.7738` | `0.5662` | Scartato come submission singola |
| `model3_obs_only_top220_no_te` (Singolo, No TE) | `0.6988` | `± 0.0046` | `0.7714` | `0.5692` | Scartato come submission singola |
| `model2_conservative_depth4_pruned_te` (*ex Submission 2*) | `0.7008` | `± 0.0024` | `0.7748` | `0.5657` | Superato dall'Ensemble |
| `model4_compact_top140_te` (*ex Submission 1*) | `0.7020` | `± 0.0036` | `0.7752` | `0.5647` | Superato dall'Ensemble |
| **Diverse 4-Model XGBoost Ensemble (`submission.csv`)** | **`0.7030`** | **`± 0.0017`** | **`0.7767`** | **`0.5636`** | **Selezionato come Unico Output** |

### Motivazione Tecnica della Scelta Unica
1. **Superiorità su Tutte le Metriche OOF**: L'ensemble pesato (`0.15 · M1 + 0.35 · M2 + 0.10 · M3 + 0.40 · M4`) supera sistematicamente ciascun modello singolo sia in **Accuracy (`0.7030`)**, sia in **ROC-AUC (`0.7767`)**, sia in **Log-Loss (`0.5636`)**.
2. **Dimezzamento della Varianza tra i Fold (`± 0.0017`)**: La combinazione di alberi con profondità diverse (`max_depth=4` vs `5`), sottoinsiemi di feature differenti (`140`, `200`, `220`, `511`) e presenza/assenza di Target Encoding stabilizza le probabilità predette sulle battaglie incerte vicine alla soglia `0.50`, proteggendo dal rischio di overfitting sul Private Leaderboard.
3. **Pipeline Diretta senza File Ridondanti**: Rimuovendo la generazione intermedia di `submission_1` e `submission_2`, la pipeline restituisce direttamente l'unico file **`submission.csv`** nella root/working directory, garantendo l'invio automatico immediato alla competizione Kaggle.

In [43]:
# ==============================================================================
# 1. TOP 25 FEATURE IMPORTANCE & DIAGNOSTIC PLOTS (via native Booster.get_score & xgb.plot_importance)
# ==============================================================================
cv_m4 = cv_results_map["model4_compact_top140_te"]
fi_df = get_feature_importance_df(cv_m4.models, cv_m4.feature_names)
print("Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):")
print(fi_df.head(25).to_string(index=False))

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    top_n = min(25, len(fi_df))
    xgb.plot_importance(
        cv_m4.models[0].get_booster(),
        ax=axes[0],
        max_num_features=top_n,
        importance_type="gain",
        title=f"Top {top_n} Feature per Gain (xgb.plot_importance)",
        xlabel="Gain",
        show_values=False,
        grid=False,
    )

    fold_ids = [f"Fold {i}" for i in range(1, FINAL_FOLDS + 1)]
    axes[1].bar(fold_ids, ens_fold_accs, color="#2e8b57", alpha=0.85, label="Fold Val Acc")
    axes[1].axhline(final_oof_acc, color="#d9534f", linestyle="--", linewidth=2, label=f"Blended OOF Acc ({final_oof_acc:.4f})")
    axes[1].set_ylim(min(ens_fold_accs) - 0.03, max(ens_fold_accs) + 0.03)
    axes[1].set_title(f"Accuratezza di Validazione sui {FINAL_FOLDS} Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()
except ImportError:
    pass

# ==============================================================================
# 2. GENERATE & VALIDATE THE SINGLE KAGGLE SUBMISSION FILE (submission.csv)
#    Uses `src.submission.generate_submission` and `validate_submission_df`
# ==============================================================================
binary_test_preds = (blended_test_probs >= DECISION_THRESHOLD).astype(int)
submission_df = generate_submission(
    test_battle_ids=test_ids,
    test_predictions=binary_test_preds,
    sample_sub_df=sample_sub,
    output_path="submission.csv",
)
sub_info = validate_submission_df(submission_df, sample_sub)

print("=" * 72)
print("File 'submission.csv' generato e validato con successo!")
print(f"Totale battaglie predette : {sub_info['n_rows']}")
print(f"Vittorie Player 1 (1)     : {sub_info['positive_count']} ({sub_info['positive_rate']*100:.2f}%)")
print(f"Vittorie Player 2 (0)     : {sub_info['negative_count']} ({(1 - sub_info['positive_rate'])*100:.2f}%)")
print("=" * 72)

display(submission_df.head(10))

Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):
                      feature  importance      gain  total_gain  weight      cover
inter_combined_material_score    0.065504 77.629081 9227.775684   118.9 763.666467
          state_hp_ratio_soft    0.029969 35.365382 3058.989526    84.0 587.552243
           state_hp_diff_soft    0.029716 35.264352 2903.762036    82.9 594.482437
          temp_hp_adv_adj_t30    0.024554 29.328138 1226.822772    42.4 371.288937
           state_hp_ratio_adj    0.018555 22.173096 1019.518341    44.3 447.250723
          state_clean_hp_diff    0.014639 17.325995 2086.847192   120.6 455.375638
            state_spikes_diff    0.013088 15.518778 1132.833813    73.2 648.043616
   temp_last5_hp_adv_adj_mean    0.012301 14.683993  846.889877    57.8 300.077238
  inter_status_x_remaining_hp    0.011439 13.555448 2153.895593   158.9 468.242188
      inter_hp_raw_x_fnt_diff    0.010888 12.935824  647.933597    50.5 322.890202
       stat

,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1
5,5,0
6,6,1
7,7,0
8,8,0
9,9,0


## Step 11: Valutazione e Sanity Check delle Predizioni contro `test.jsonl`

Poiché nel dataset di test del torneo Kaggle (`test.jsonl`) la colonna target reale (`player_won`) è assente (oscurata dagli organizzatori per la valutazione sulla Leaderboard privata), eseguiamo una **valutazione diagnostica approfondita e consistenza strutturale** incrociando le predizioni generate in `submission.csv` con la telemetria osservata in `test.jsonl`:

1. **Integrità e Allineamento 1-a-1**: Verifica dell'ordine esatto e corrispondenza biunivoca dei `battle_id`.
2. **Distribuzione delle Probabilità e Confidenza**: Percentili, entropia e dispersione di calibrazione delle probabilità ensemble `blended_test_probs`.
3. **Coerenza con lo Stato Finale della Battaglia (Turno 30)**:
   - Correlazione con il differenziale HP (stimato e osservato)
   - Tasso di vittoria predetto quando P1 ha un vantaggio netto di HP ($> +20\%$ e $> +50\%$)
   - Tasso di vittoria predetto quando P2 ha un vantaggio netto di HP ($< -20\%$ e $< -50\%$)
4. **Analisi dei Casi Estremi (Borderline vs Strong Advantage)**: Ispezione qualitativa dei casi ad alta confidenza ($p > 0.85$ e $p < 0.15$) e dei match in bilico ($0.48 \le p \le 0.52$).

In [47]:
# ==============================================================================
# STEP 11: VALUTAZIONE DIAGNOSTICA DELLA SUBMISSION CONTRO TEST.JSONL
# ==============================================================================
print("=" * 80)
print("VALUTAZIONE DIAGNOSTICA DELLA SUBMISSION RISPETTO A TEST.JSONL")
print("=" * 80)

# 1. Check if target variable exists in test_battles
test_has_target = any("player_won" in b for b in test_battles)

if test_has_target:
  y_true_test = np.array([int(b["player_won"]) for b in test_battles])
  acc_test = accuracy_score(y_true_test, submission_df["player_won"].values)
  ll_test = log_loss(y_true_test, blended_test_probs)
  auc_test = roc_auc_score(y_true_test, blended_test_probs)
  print(
      f"[GROUND TRUTH DISPONIBILE] Test Acc: {acc_test:.4f} | LogLoss:"
      f" {ll_test:.4f} | ROC-AUC: {auc_test:.4f}"
  )
else:
  print("[KAGGLE HIDDEN TEST] Target 'player_won' non presente nel test set.\n")

  # 2. Probability Calibration & Distribution Telemetry
  print(
      f"• Mean predicted probability (P1): {blended_test_probs.mean():.4f} (std:"
      f" {blended_test_probs.std():.4f})"
  )
  print(
      f"• 10th percentile: {np.percentile(blended_test_probs, 10):.4f} | 90th"
      f" percentile: {np.percentile(blended_test_probs, 90):.4f}"
  )

  # 3. Turn 30 HP Advantage Alignment Sanity Check
  hp_diff_t30 = X_test_full["state_hp_diff_raw"].values
  p1_net_lead = hp_diff_t30 > 0.20
  p2_net_lead = hp_diff_t30 < -0.20

  winrate_p1_lead = submission_df.loc[p1_net_lead, "player_won"].mean()
  winrate_p2_lead = submission_df.loc[p2_net_lead, "player_won"].mean()

  print(
      f"• Predicted P1 win rate with heavy HP lead (>+20%):"
      f" {winrate_p1_lead * 100:.2f}%"
  )
  print(
      f"• Predicted P1 win rate with heavy HP deficit (<-20%):"
      f" {winrate_p2_lead * 100:.2f}%"
  )

  # 4. Borderline Match Ratio
  borderline_mask = (blended_test_probs >= 0.45) & (blended_test_probs <= 0.55)
  print(
      f"• Borderline matches (0.45 <= P <= 0.55): {borderline_mask.mean() * 100:.2f}%"
  )

VALUTAZIONE DIAGNOSTICA DELLA SUBMISSION RISPETTO A TEST.JSONL
[KAGGLE HIDDEN TEST] Target 'player_won' non presente nel test set.

• Mean predicted probability (P1): 0.5042 (std: 0.2525)
• 10th percentile: 0.1441 | 90th percentile: 0.8441
• Predicted P1 win rate with heavy HP lead (>+20%): 92.76%
• Predicted P1 win rate with heavy HP deficit (<-20%): 29.93%
• Borderline matches (0.45 <= P <= 0.55): 12.80%


In [46]:
# ==============================================================================
# 1. TOP 25 FEATURE IMPORTANCE & DIAGNOSTIC PLOTS (via native Booster.get_score & xgb.plot_importance)
# ==============================================================================
cv_m4 = cv_results_map["model4_compact_top140_te"]
fi_df = get_feature_importance_df(cv_m4.models, cv_m4.feature_names)
print("Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):")
print(fi_df.head(25).to_string(index=False))

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    top_n = min(25, len(fi_df))
    xgb.plot_importance(
        cv_m4.models[0].get_booster(),
        ax=axes[0],
        max_num_features=top_n,
        importance_type="gain",
        title=f"Top {top_n} Feature per Gain (xgb.plot_importance)",
        xlabel="Gain",
        show_values=False,
        grid=False,
    )

    fold_ids = [f"Fold {i}" for i in range(1, FINAL_FOLDS + 1)]
    axes[1].bar(fold_ids, ens_fold_accs, color="#2e8b57", alpha=0.85, label="Fold Val Acc")
    axes[1].axhline(final_oof_acc, color="#d9534f", linestyle="--", linewidth=2, label=f"Blended OOF Acc ({final_oof_acc:.4f})")
    axes[1].set_ylim(min(ens_fold_accs) - 0.03, max(ens_fold_accs) + 0.03)
    axes[1].set_title(f"Accuratezza di Validazione sui {FINAL_FOLDS} Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()
except ImportError:
    pass

# ==============================================================================
# 2. GENERATE & VALIDATE THE SINGLE KAGGLE SUBMISSION FILE (submission.csv)
#    Uses `src.submission.generate_submission` and `validate_submission_df`
# ==============================================================================
binary_test_preds = (blended_test_probs >= DECISION_THRESHOLD).astype(int)
submission_df = generate_submission(
    test_battle_ids=test_ids,
    test_predictions=binary_test_preds,
    sample_sub_df=sample_sub,
    output_path="submission.csv",
)
sub_info = validate_submission_df(submission_df, sample_sub)

print("=" * 72)
print("File 'submission.csv' generato e validato con successo!")
print(f"Totale battaglie predette : {sub_info['n_rows']}")
print(f"Vittorie Player 1 (1)     : {sub_info['positive_count']} ({sub_info['positive_rate']*100:.2f}%)")
print(f"Vittorie Player 2 (0)     : {sub_info['negative_count']} ({(1 - sub_info['positive_rate'])*100:.2f}%)")
print("=" * 72)

display(submission_df.head(10))

Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):
                      feature  importance      gain  total_gain  weight      cover
inter_combined_material_score    0.065504 77.629081 9227.775684   118.9 763.666467
          state_hp_ratio_soft    0.029969 35.365382 3058.989526    84.0 587.552243
           state_hp_diff_soft    0.029716 35.264352 2903.762036    82.9 594.482437
          temp_hp_adv_adj_t30    0.024554 29.328138 1226.822772    42.4 371.288937
           state_hp_ratio_adj    0.018555 22.173096 1019.518341    44.3 447.250723
          state_clean_hp_diff    0.014639 17.325995 2086.847192   120.6 455.375638
            state_spikes_diff    0.013088 15.518778 1132.833813    73.2 648.043616
   temp_last5_hp_adv_adj_mean    0.012301 14.683993  846.889877    57.8 300.077238
  inter_status_x_remaining_hp    0.011439 13.555448 2153.895593   158.9 468.242188
      inter_hp_raw_x_fnt_diff    0.010888 12.935824  647.933597    50.5 322.890202
       stat

,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1
5,5,0
6,6,1
7,7,0
8,8,0
9,9,0
